# 📈 Notebook 1: Understanding Write Bottlenecks

Before scaling, we need to understand what makes writes different from reads and identify where bottlenecks occur.

## Learning Objectives

By the end of this notebook, you'll understand:
- How writes differ from reads
- Where write bottlenecks occur
- How to measure write throughput
- When write scaling is actually needed

## 🛠️ Setup

```bash
cd patterns/scaling-writes
docker-compose up -d
```

🔍 **Open Adminer** at http://localhost:8080 to watch write operations!

In [1]:
import psycopg2
import time
import statistics
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

✅ Connected to PostgreSQL


## 📊 Reads vs Writes: The Fundamental Difference

In [2]:
print("📊 Reads vs Writes: What Makes Them Different")
print("=" * 60)
print("""
READS:
─────────────────────────────────────────────────────────────
• Can be served from cache/memory
• Can be parallelized across replicas
• Stateless - doesn't change data
• Failure = retry (no side effects)

WRITES:
─────────────────────────────────────────────────────────────
• Must go to disk (durability)
• Must go to single leader (consistency)
• Stateful - changes data permanently
• Failure = complex (partial writes?)
• Requires: locks, indexes, replication

THE ASYMMETRY:
─────────────────────────────────────────────────────────────
• Adding read replicas: Easy! Just copy data.
• Adding write capacity: Hard! Must coordinate.

This is why write scaling requires different strategies!
""")

📊 Reads vs Writes: What Makes Them Different

READS:
─────────────────────────────────────────────────────────────
• Can be served from cache/memory
• Can be parallelized across replicas
• Stateless - doesn't change data
• Failure = retry (no side effects)

WRITES:
─────────────────────────────────────────────────────────────
• Must go to disk (durability)
• Must go to single leader (consistency)
• Stateful - changes data permanently
• Failure = complex (partial writes?)
• Requires: locks, indexes, replication

THE ASYMMETRY:
─────────────────────────────────────────────────────────────
• Adding read replicas: Easy! Just copy data.
• Adding write capacity: Hard! Must coordinate.

This is why write scaling requires different strategies!



## 🔬 Measuring Write Throughput

In [3]:
def single_write() -> float:
    conn = get_connection()
    cursor = conn.cursor()
    
    start = time.time()
    cursor.execute(
        "INSERT INTO events (event_type, user_id, payload) VALUES (%s, %s, %s)",
        ('click', 1, '{"page": "home"}')
    )
    conn.commit()
    elapsed = time.time() - start
    
    conn.close()
    return elapsed * 1000

print("🔬 Measuring Single Write Latency")
print("=" * 60)

times = [single_write() for _ in range(100)]

print(f"\nSingle write statistics (100 writes):")
print(f"   Mean:   {statistics.mean(times):.2f}ms")
print(f"   Median: {statistics.median(times):.2f}ms")
print(f"   P95:    {sorted(times)[95]:.2f}ms")
print(f"   Max:    {max(times):.2f}ms")

writes_per_sec = 1000 / statistics.mean(times)
print(f"\n📊 Estimated throughput: ~{writes_per_sec:.0f} writes/sec (single thread)")

🔬 Measuring Single Write Latency



Single write statistics (100 writes):
   Mean:   2.05ms
   Median: 1.82ms
   P95:    3.26ms
   Max:    3.88ms

📊 Estimated throughput: ~488 writes/sec (single thread)


In [4]:
def write_worker(num_writes: int) -> list:
    times = []
    conn = get_connection()
    cursor = conn.cursor()
    
    for i in range(num_writes):
        start = time.time()
        cursor.execute(
            "INSERT INTO events (event_type, user_id, payload) VALUES (%s, %s, %s)",
            ('click', i, '{"page": "home"}')
        )
        conn.commit()
        times.append((time.time() - start) * 1000)
    
    conn.close()
    return times

print("🔬 Concurrent Write Throughput Test")
print("=" * 60)

for num_workers in [1, 5, 10, 20]:
    writes_per_worker = 50
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(write_worker, writes_per_worker) for _ in range(num_workers)]
        all_times = []
        for f in futures:
            all_times.extend(f.result())
    total_time = time.time() - start
    
    total_writes = num_workers * writes_per_worker
    throughput = total_writes / total_time
    
    print(f"\n👥 {num_workers} concurrent writers:")
    print(f"   Total writes: {total_writes}")
    print(f"   Total time:   {total_time:.2f}s")
    print(f"   Throughput:   {throughput:.0f} writes/sec")
    print(f"   Avg latency:  {statistics.mean(all_times):.2f}ms")

🔬 Concurrent Write Throughput Test

👥 1 concurrent writers:
   Total writes: 50
   Total time:   0.07s
   Throughput:   703 writes/sec
   Avg latency:  1.06ms

👥 5 concurrent writers:
   Total writes: 250
   Total time:   0.07s
   Throughput:   3395 writes/sec
   Avg latency:  0.85ms



👥 10 concurrent writers:
   Total writes: 500
   Total time:   0.09s
   Throughput:   5371 writes/sec
   Avg latency:  1.14ms

👥 20 concurrent writers:
   Total writes: 1000
   Total time:   0.15s
   Throughput:   6868 writes/sec
   Avg latency:  1.73ms


## 🎯 Write Bottleneck Sources

In [5]:
print("🎯 Where Write Bottlenecks Occur")
print("=" * 60)
print("""
1. DISK I/O
─────────────────────────────────────────────────────────────
   • Write-ahead log (WAL) must be flushed
   • Data pages written to disk
   • SSD: ~100,000 IOPS | HDD: ~100 IOPS

2. CPU
─────────────────────────────────────────────────────────────
   • Query parsing and planning
   • Index updates (B-tree rebalancing)
   • Constraint checking
   • Triggers and stored procedures

3. MEMORY
─────────────────────────────────────────────────────────────
   • Buffer pool for dirty pages
   • Lock management
   • Connection overhead

4. NETWORK
─────────────────────────────────────────────────────────────
   • Replication to replicas
   • Client-server round trips
   • Distributed transactions

5. CONTENTION
─────────────────────────────────────────────────────────────
   • Row locks on same data
   • Table locks
   • Index page locks
""")

🎯 Where Write Bottlenecks Occur

1. DISK I/O
─────────────────────────────────────────────────────────────
   • Write-ahead log (WAL) must be flushed
   • Data pages written to disk
   • SSD: ~100,000 IOPS | HDD: ~100 IOPS

2. CPU
─────────────────────────────────────────────────────────────
   • Query parsing and planning
   • Index updates (B-tree rebalancing)
   • Constraint checking
   • Triggers and stored procedures

3. MEMORY
─────────────────────────────────────────────────────────────
   • Buffer pool for dirty pages
   • Lock management
   • Connection overhead

4. NETWORK
─────────────────────────────────────────────────────────────
   • Replication to replicas
   • Client-server round trips
   • Distributed transactions

5. CONTENTION
─────────────────────────────────────────────────────────────
   • Row locks on same data
   • Table locks
   • Index page locks



In [6]:
print("📈 Index Overhead on Writes")
print("=" * 60)

def write_with_indexes(num_writes: int) -> float:
    conn = get_connection()
    cursor = conn.cursor()
    
    start = time.time()
    for i in range(num_writes):
        cursor.execute(
            "INSERT INTO events (event_type, user_id, payload) VALUES (%s, %s, %s)",
            ('pageview', i % 1000, '{"page": "product"}')
        )
    conn.commit()
    elapsed = time.time() - start
    
    conn.close()
    return elapsed

conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT indexname FROM pg_indexes WHERE tablename = 'events'")
indexes = cursor.fetchall()
conn.close()

print(f"\nCurrent indexes on 'events' table:")
for idx in indexes:
    print(f"   • {idx[0]}")

print(f"\n💡 Each index must be updated on every INSERT!")
print(f"   More indexes = slower writes, faster reads")
print(f"   This is the classic read/write trade-off.")

📈 Index Overhead on Writes

Current indexes on 'events' table:
   • events_pkey
   • idx_events_user_id
   • idx_events_type
   • idx_events_created

💡 Each index must be updated on every INSERT!
   More indexes = slower writes, faster reads
   This is the classic read/write trade-off.


## 🧮 Back-of-Envelope: Do You Need Write Scaling?

In [7]:
print("🧮 Back-of-Envelope Calculation")
print("=" * 60)
print("""
SCENARIO: Social media like button
─────────────────────────────────────────────────────────────

Given:
• 100 million daily active users
• Average user likes 10 posts/day
• Peak traffic = 3x average

Calculation:
• Daily likes = 100M × 10 = 1 billion likes/day
• Average per second = 1B / 86,400 ≈ 11,500 likes/sec
• Peak = 11,500 × 3 ≈ 35,000 likes/sec

Can a single PostgreSQL handle this?
─────────────────────────────────────────────────────────────
• Well-tuned PostgreSQL: ~10,000-50,000 writes/sec
• Answer: Maybe at average, NO at peak!

We need write scaling strategies!
""")

def calculate_write_needs(dau: int, actions_per_user: int, peak_multiplier: float = 3.0):
    daily_writes = dau * actions_per_user
    avg_per_second = daily_writes / 86400
    peak_per_second = avg_per_second * peak_multiplier
    
    print(f"\n📊 Your scenario:")
    print(f"   Daily writes:    {daily_writes:,.0f}")
    print(f"   Average/sec:     {avg_per_second:,.0f}")
    print(f"   Peak/sec:        {peak_per_second:,.0f}")
    
    if peak_per_second < 1000:
        print(f"   ✅ Single DB can handle this easily")
    elif peak_per_second < 10000:
        print(f"   ⚠️ Need optimization, maybe single DB")
    else:
        print(f"   ❌ Need sharding/batching strategies")

calculate_write_needs(dau=1_000_000, actions_per_user=5)
calculate_write_needs(dau=100_000_000, actions_per_user=10)

🧮 Back-of-Envelope Calculation

SCENARIO: Social media like button
─────────────────────────────────────────────────────────────

Given:
• 100 million daily active users
• Average user likes 10 posts/day
• Peak traffic = 3x average

Calculation:
• Daily likes = 100M × 10 = 1 billion likes/day
• Average per second = 1B / 86,400 ≈ 11,500 likes/sec
• Peak = 11,500 × 3 ≈ 35,000 likes/sec

Can a single PostgreSQL handle this?
─────────────────────────────────────────────────────────────
• Well-tuned PostgreSQL: ~10,000-50,000 writes/sec
• Answer: Maybe at average, NO at peak!

We need write scaling strategies!


📊 Your scenario:
   Daily writes:    5,000,000
   Average/sec:     58
   Peak/sec:        174
   ✅ Single DB can handle this easily

📊 Your scenario:
   Daily writes:    1,000,000,000
   Average/sec:     11,574
   Peak/sec:        34,722
   ❌ Need sharding/batching strategies


## 🧪 Quick Quiz

1. **Why can't you just add write replicas like read replicas?**

2. **What happens to write throughput as you add more indexes?**

3. **When should you NOT worry about write scaling?**

In [8]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why no write replicas:")
print("   - Writes must be coordinated (consistency)")
print("   - Multi-master = conflict resolution nightmare")
print("   - Read replicas receive writes FROM leader")
print()
print("2. More indexes = slower writes:")
print("   - Each INSERT updates ALL indexes")
print("   - B-tree rebalancing has overhead")
print("   - Classic read/write trade-off")
print()
print("3. When NOT to worry:")
print("   - < 1,000 writes/sec (most apps!)")
print("   - When reads are the actual bottleneck")
print("   - Before you've done the math!")

📝 Quiz Answers

1. Why no write replicas:
   - Writes must be coordinated (consistency)
   - Multi-master = conflict resolution nightmare
   - Read replicas receive writes FROM leader

2. More indexes = slower writes:
   - Each INSERT updates ALL indexes
   - B-tree rebalancing has overhead
   - Classic read/write trade-off

3. When NOT to worry:
   - < 1,000 writes/sec (most apps!)
   - When reads are the actual bottleneck
   - Before you've done the math!


## 📚 Summary

### Key Takeaways

1. **Writes are harder to scale than reads** - Must coordinate, can't just replicate
2. **Do the math first** - Many apps don't need write scaling
3. **Bottlenecks vary** - Disk I/O, CPU, contention all matter
4. **Indexes help reads, hurt writes** - Know your trade-offs
5. **Peak matters more than average** - Design for bursts

### Next Up

In **Notebook 2**, we'll learn database optimization for writes:
- Write-optimized databases
- Index management
- WAL tuning